In [1]:
from dotenv import load_dotenv
import os
api_key = os.getenv("ROBOFLOW_API_KEY")

In [2]:
BATCH_SIZE = 16

In [3]:
NEW_NUM_QUERIES = 50

In [4]:
# !pip install roboflow transformers

In [5]:
# !pip install typing_extensions==4.15.0 torch==2.7.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

In [6]:
from roboflow import Roboflow
rf = Roboflow(api_key=api_key)
project = rf.workspace("crater-zqpjg").project("crater-vrqmn")
version = project.version(1)
dataset = version.download("coco")

loading Roboflow workspace...
loading Roboflow project...


In [7]:
dataset_path = "./crater-1"

In [8]:
import os

for split in ["train", "valid", "test"]:
    images = os.listdir(f"{dataset_path}/{split}")
    print(f"{split}: {len(images)} images")

train: 2490 images
valid: 712 images
test: 357 images


In [9]:
import torch
import json
import os
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import AutoImageProcessor

processor = AutoImageProcessor.from_pretrained("facebook/detr-resnet-50")

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


In [10]:
class COCODataset(Dataset):
    def __init__(self, images_dir, annotations_file, augment=False):
        self.images_dir = images_dir
        self.augment = augment

        # Load COCO annotations JSON
        with open(annotations_file, "r") as f:
            coco = json.load(f)

        # Map image_id -> image info
        self.images = {img["id"]: img for img in coco["images"]}
        self.image_ids = list(self.images.keys())

        # Map image_id -> list of annotations
        self.annotations = {}
        for ann in coco["annotations"]:
            img_id = ann["image_id"]
            self.annotations.setdefault(img_id, []).append(ann)

        self.categories = {cat["id"]: cat["name"] for cat in coco["categories"]}

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        image_info = self.images[image_id]

        # Load image
        img_path = os.path.join(self.images_dir, image_info["file_name"])
        image = Image.open(img_path).convert("RGB")

        # Get annotations for this image
        anns = self.annotations.get(image_id, [])

        coco_annotations = []
        for ann in anns:
            coco_annotations.append({
                "id":          ann["id"],
                "image_id":    image_id,
                "category_id": ann["category_id"] - 1,  # 1 → 0
                "bbox":        ann["bbox"],
                "area":        ann["area"],
                "iscrowd":     ann.get("iscrowd", 0),
            })

        target = {
            "image_id":    image_id,
            "annotations": coco_annotations,
        }

        # Augmentation on PIL image before processor
        if self.augment:
            import torchvision.transforms.functional as TF
            import random
            if random.random() > 0.5:
                image = TF.hflip(image)
                # Flip boxes too: new_x = img_w - x - w
                w_img = image.width
                for ann in target["annotations"]:
                    x, y, w, h = ann["bbox"]
                    ann["bbox"] = [w_img - x - w, y, w, h]

        return image, target

In [11]:
train_dataset = COCODataset(
    images_dir=f"{dataset_path}/train",
    annotations_file=f"{dataset_path}/train/_annotations.coco.json",
    augment=True
)

valid_dataset = COCODataset(
    images_dir=f"{dataset_path}/valid",
    annotations_file=f"{dataset_path}/valid/_annotations.coco.json",
    augment=False
)

test_dataset = COCODataset(
    images_dir=f"{dataset_path}/test",
    annotations_file=f"{dataset_path}/test/_annotations.coco.json",
    augment=False
)

In [12]:
def collate_fn(batch):
    images, targets = zip(*batch)

    # Processor handles resize, normalize, padding, and pixel_mask
    encoding = processor(
        images=list(images),
        annotations=list(targets),
        return_tensors="pt"
    )
    return encoding

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

In [13]:
batch = next(iter(train_loader))
print(f"pixel_values shape : {batch['pixel_values'].shape}")   # [B, 3, H, W]
print(f"pixel_mask shape   : {batch['pixel_mask'].shape}")     # [B, H, W]
print(f"labels[0] keys     : {batch['labels'][0].keys()}")
print(f"boxes (img 0)      : {batch['labels'][0]['boxes']}")   # normalised [cx,cy,w,h]
print(f"class_labels (img 0): {batch['labels'][0]['class_labels']}")

pixel_values shape : torch.Size([16, 3, 800, 800])
pixel_mask shape   : torch.Size([16, 800, 800])
labels[0] keys     : KeysView({'size': tensor([800, 800]), 'image_id': tensor([2319]), 'class_labels': tensor([0, 0, 0, 0]), 'boxes': tensor([[0.3954, 0.6430, 0.5312, 0.5312],
        [0.7764, 0.2921, 0.1442, 0.1418],
        [0.4892, 0.8762, 0.0601, 0.0601],
        [0.5288, 0.7716, 0.0529, 0.0529]]), 'area': tensor([180625.0000,  13091.7158,   2311.3904,   1789.9408]), 'iscrowd': tensor([0, 0, 0, 0]), 'orig_size': tensor([416, 416])})
boxes (img 0)      : tensor([[0.3954, 0.6430, 0.5312, 0.5312],
        [0.7764, 0.2921, 0.1442, 0.1418],
        [0.4892, 0.8762, 0.0601, 0.0601],
        [0.5288, 0.7716, 0.0529, 0.0529]])
class_labels (img 0): tensor([0, 0, 0, 0])


In [14]:
import json

def count_object_sizes(annotations_file):
    with open(annotations_file) as f:
        coco = json.load(f)

    small, medium, large = 0, 0, 0

    for ann in coco["annotations"]:
        w, h = ann["bbox"][2], ann["bbox"][3]
        area = w * h

        if area < 32**2:
            small += 1
        elif area < 96**2:
            medium += 1
        else:
            large += 1

    total = small + medium + large

    print(f"Total objects : {total}")
    print(f"Small         : {small}  ({100*small/total:.1f}%)")
    print(f"Medium        : {medium} ({100*medium/total:.1f}%)")
    print(f"Large         : {large}  ({100*large/total:.1f}%)")

    return {"small": small, "medium": medium, "large": large, "total": total}

In [15]:
for split in ["train", "valid", "test"]:
    print(f"\n── {split.upper()} ──")
    count_object_sizes(f"{dataset_path}/{split}/_annotations.coco.json")


── TRAIN ──
Total objects : 4953
Small         : 3243  (65.5%)
Medium        : 1378 (27.8%)
Large         : 332  (6.7%)

── VALID ──
Total objects : 1364
Small         : 862  (63.2%)
Medium        : 431 (31.6%)
Large         : 71  (5.2%)

── TEST ──
Total objects : 731
Small         : 467  (63.9%)
Medium        : 219 (30.0%)
Large         : 45  (6.2%)


In [16]:
from transformers import AutoModelForObjectDetection

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# DETR num_labels = actual classes only, NO background token
# Dataset has 1 class (crater), so num_labels=1
NUM_CLASSES = 1
id2label = {0: "crater"}
label2id = {"crater": 0}

model_name = "facebook/detr-resnet-50"

base_model = AutoModelForObjectDetection.from_pretrained(model_name).to(device)
base_query_weights = base_model.model.query_position_embeddings.weight.data.clone()

model = AutoModelForObjectDetection.from_pretrained(
    model_name,
    num_labels=NUM_CLASSES,
    id2label=id2label,
    label2id=label2id,
    num_queries=NEW_NUM_QUERIES,
    ignore_mismatched_sizes=True
).to(device)
# Perform Weight Surgery
with torch.no_grad():
    if NEW_NUM_QUERIES < 100:
        # Take the first N queries
        model.model.query_position_embeddings.weight.data = base_query_weights[:NEW_NUM_QUERIES]
        
    elif NEW_NUM_QUERIES > 100:
        # Repeat the queries to fill the new capacity
        repeats = NEW_NUM_QUERIES // 100
        remainder = NEW_NUM_QUERIES % 100
        
        new_weights = torch.cat([
            base_query_weights.repeat(repeats, 1), 
            base_query_weights[:remainder]
        ])
        model.model.query_position_embeddings.weight.data = new_weights

Using device: cuda


Loading weights: 100%|██████████| 530/530 [00:00<00:00, 14478.57it/s]
[transformers] DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] You passed `num_labels=1` which is incompatible to the `id2label` map of length `91`.
Loading weights: 100%|██████████| 530/530 [00:00<00:00, 12111.57it/s]
[transformers] DetrForObjectDetection LOAD REPORT from: facebook/d

In [17]:
param_groups = [
    {"params": [p for n, p in model.named_parameters() if "backbone" in n], "lr": 1e-5},
    {"params": [p for n, p in model.named_parameters() if "backbone" not in n], "lr": 1e-4},
]

optimizer = torch.optim.AdamW(param_groups, weight_decay=1e-4)

# Cosine LR decay — works better than StepLR for transformers
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=50,   # set to NUM_EPOCHS below
    eta_min=1e-6
)

In [18]:
def train_one_epoch(model, optimizer, dataloader, device, epoch):
    model.train()
    total_loss = 0

    for batch_idx, batch in enumerate(dataloader):
        pixel_values = batch["pixel_values"].to(device)
        pixel_mask   = batch["pixel_mask"].to(device)
        labels       = [{k: v.to(device) for k, v in t.items()} for t in batch["labels"]]

        outputs = model(
            pixel_values=pixel_values,
            pixel_mask=pixel_mask,
            labels=labels
        )

        loss = outputs.loss
        loss_dict = outputs.loss_dict   # cls, bbox, giou components

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.1)  # standard for DETR
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 10 == 0:
            print(f"Epoch [{epoch}] Batch [{batch_idx}/{len(dataloader)}] "
                  f"Loss: {loss.item():.4f} "
                  f"(cls: {loss_dict['loss_ce'].item():.4f}, "
                  f"bbox: {loss_dict['loss_bbox'].item():.4f}, "
                  f"giou: {loss_dict['loss_giou'].item():.4f})")

    return total_loss / len(dataloader)

In [19]:
@torch.no_grad()
def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0

    for batch in dataloader:
        pixel_values = batch["pixel_values"].to(device)
        pixel_mask   = batch["pixel_mask"].to(device)
        labels       = [{k: v.to(device) for k, v in t.items()} for t in batch["labels"]]

        # DETR returns loss in eval mode too when labels are passed — no train() trick needed
        outputs = model(
            pixel_values=pixel_values,
            pixel_mask=pixel_mask,
            labels=labels
        )
        total_loss += outputs.loss.item()

    return total_loss / len(dataloader)

In [20]:
NUM_EPOCHS = 50
best_val_loss = float("inf")

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(model, optimizer, train_loader, device, epoch)
    val_loss   = evaluate(model, valid_loader, device)
    lr_scheduler.step()

    print(f"\nEpoch [{epoch}/{NUM_EPOCHS}] "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f}\n")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), f"best_detr_nq{NEW_NUM_QUERIES}.pth")
        print(f"  ✅ Saved best model (val_loss: {val_loss:.4f})")

print("Training complete!")

Epoch [1] Batch [0/156] Loss: 5.1527 (cls: 0.6801, bbox: 0.4756, giou: 1.0473)
Epoch [1] Batch [10/156] Loss: 1.8206 (cls: 0.5535, bbox: 0.0819, giou: 0.4288)
Epoch [1] Batch [20/156] Loss: 1.8145 (cls: 0.5277, bbox: 0.0598, giou: 0.4938)
Epoch [1] Batch [30/156] Loss: 2.1522 (cls: 0.5285, bbox: 0.1137, giou: 0.5276)
Epoch [1] Batch [40/156] Loss: 1.7723 (cls: 0.5648, bbox: 0.0685, giou: 0.4325)
Epoch [1] Batch [50/156] Loss: 1.8998 (cls: 0.5003, bbox: 0.0916, giou: 0.4707)
Epoch [1] Batch [60/156] Loss: 1.7351 (cls: 0.4685, bbox: 0.0569, giou: 0.4911)
Epoch [1] Batch [70/156] Loss: 1.5365 (cls: 0.4795, bbox: 0.0470, giou: 0.4111)
Epoch [1] Batch [80/156] Loss: 1.8791 (cls: 0.4727, bbox: 0.0625, giou: 0.5469)
Epoch [1] Batch [90/156] Loss: 1.7144 (cls: 0.4539, bbox: 0.0530, giou: 0.4979)
Epoch [1] Batch [100/156] Loss: 1.6226 (cls: 0.3719, bbox: 0.0445, giou: 0.5142)
Epoch [1] Batch [110/156] Loss: 1.2602 (cls: 0.2885, bbox: 0.0471, giou: 0.3682)
Epoch [1] Batch [120/156] Loss: 1.4059 

In [20]:
def _cxcywh_to_xyxy_pixels(boxes, orig_size):
    """Convert normalised [cx,cy,w,h] to pixel [x1,y1,x2,y2]."""
    h, w = orig_size[0].item(), orig_size[1].item()
    cx, cy, bw, bh = boxes.unbind(-1)
    x1 = (cx - bw / 2) * w
    y1 = (cy - bh / 2) * h
    x2 = (cx + bw / 2) * w
    y2 = (cy + bh / 2) * h
    return torch.stack([x1, y1, x2, y2], dim=-1)

In [21]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision

@torch.no_grad()
def evaluate_map(model, dataloader, device):
    model.eval()

    metric = MeanAveragePrecision(
        iou_type="bbox",
        iou_thresholds=[0.5, 0.75],
        max_detection_thresholds=[1, 10, 100]
    )

    for batch in dataloader:
        pixel_values = batch["pixel_values"].to(device)
        pixel_mask   = batch["pixel_mask"].to(device)
        orig_sizes   = torch.stack([
            torch.tensor([t["orig_size"][0], t["orig_size"][1]]) for t in batch["labels"]
        ]).to(device)

        outputs = model(pixel_values=pixel_values, pixel_mask=pixel_mask)

        # Post-process converts normalised [cx,cy,w,h] → [x1,y1,x2,y2] in pixel coords
        results = processor.post_process_object_detection(
            outputs,
            threshold=0,
            target_sizes=orig_sizes
        )

        preds = [{
            "boxes":  r["boxes"].cpu(),
            "scores": r["scores"].cpu(),
            "labels": r["labels"].cpu(),
        } for r in results]

        gts = [{
            # labels store boxes as normalised cxcywh — convert back to xyxy pixels
            "boxes":  _cxcywh_to_xyxy_pixels(t["boxes"], t["orig_size"]).cpu(),
            "labels": t["class_labels"].cpu(),
        } for t in batch["labels"]]

        metric.update(preds, gts)

    return metric.compute()

In [22]:
model.load_state_dict(torch.load(f"best_detr_nq{NEW_NUM_QUERIES}.pth", map_location=device))
results = evaluate_map(model, test_loader, device)

# --- 1. Compute F1-Score ---
# Get the overall mAP and the mAR at max 100 detections
map_val = results.get("map", torch.tensor(0.0)).item()
mar_val = results.get("mar_100", torch.tensor(0.0)).item()

# Calculate harmonic mean for F1-score. Add a tiny epsilon (1e-8) to avoid division by zero.
if (map_val + mar_val) > 0:
    f1_score = 2 * (map_val * mar_val) / (map_val + mar_val + 1e-8)
else:
    f1_score = 0.0

# --- 2. Define all metrics to print ---
metrics_to_print = {
    # Precision Metrics
    "mAP@0.50:0.95 (all)"      : "map",
    "mAP@0.50      (all)"      : "map_50",
    "mAP@0.75      (all)"      : "map_75",
    "mAP@0.50:0.95 (small)"    : "map_small",
    "mAP@0.50:0.95 (medium)"   : "map_medium",
    "mAP@0.50:0.95 (large)"    : "map_large",
    
    # Recall Metrics (corresponds to your max_detection_thresholds)
    "mAR@0.50:0.95 (max=1)"    : "mar_1",
    "mAR@0.50:0.95 (max=10)"   : "mar_10",
    "mAR@0.50:0.95 (max=100)"  : "mar_100",
    "mAR@0.50:0.95 (small)"    : "mar_small",
    "mAR@0.50:0.95 (medium)"   : "mar_medium",
    "mAR@0.50:0.95 (large)"    : "mar_large",
}

# --- 3. Print the table ---
print("=" * 45)
print(f"{'Metric':<30} {'Value':>10}")
print("=" * 45)

# Loop through and print mAP and mAR
for label, key in metrics_to_print.items():
    val = results.get(key, torch.tensor(float("nan"))).item()
    print(f"{label:<30} {val:>10.4f}")

print("-" * 45)
# Print the custom F1-score
print(f"{'F1-Score (mAP & mAR@100)':<30} {f1_score:>10.4f}")
print("=" * 45)

Metric                              Value
mAP@0.50:0.95 (all)                0.6672
mAP@0.50      (all)                0.9017
mAP@0.75      (all)                0.4326
mAP@0.50:0.95 (small)              0.5445
mAP@0.50:0.95 (medium)             0.8398
mAP@0.50:0.95 (large)              0.9531
mAR@0.50:0.95 (max=1)              0.3988
mAR@0.50:0.95 (max=10)             0.7702
mAR@0.50:0.95 (max=100)            0.7791
mAR@0.50:0.95 (small)              0.7105
mAR@0.50:0.95 (medium)             0.8939
mAR@0.50:0.95 (large)              0.9667
---------------------------------------------
F1-Score (mAP & mAR@100)           0.7188
